# Convert an extract of PyPSA-Eur into GEMS format (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AntaresSimulatorTeam/GEMS/blob/main/doc/examples/notebooks/tutorial-two-pypsa-eur/tutorial_pypsa_eur_colab.ipynb)

This notebook walks through the PyPSA-Eur workflow: getting a PyPSA-Eur network, converting it to a GEMS study, running it with GemsPy and checking the result against PyPSA.

## Part 0 - Setup and import of the PyPSA-Eur network

Only one PyPSA-Eur output is needed: `base_s_1_elec.nc`. The cell below either **rebuilds it with Pixi** (Snakemake runs only up to this file) or **downloads the pre-generated copy** (default, a few seconds).

The first cell installs the Python packages on Colab. If Colab then asks to **restart the session** (some pre-installed packages were upgraded), restart it and run the cells again from the top.

In [ ]:
# Colab only: install the Python packages used by the tutorial (no-op elsewhere)
import subprocess, sys
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pypsa", "netcdf4", "xarray", "matplotlib",
                    "pypsa-to-gems-converter==0.0.2", "gemspy==0.3.0"], check=True)
    print("Packages installed.")
else:
    print("Not running on Colab: packages are expected to be installed already.")

In [ ]:
import os, shutil, subprocess
from pathlib import Path

WORKDIR = Path("/content") if "google.colab" in sys.modules else Path.cwd()
os.chdir(WORKDIR)

# The tutorial needs ONE file only: the 1-node France electricity network (a few hundred kB).
network_file = Path("pypsa-eur/resources/networks/base_s_1_elec.nc")

# True  -> rebuild it with Pixi + Snakemake (installs a ~3.5 GB env and downloads ~10 GB of
#          PyPSA-Eur data, incl. a weather cutout: about 30-60 min on Colab).
# False -> download the pre-generated copy shipped with the GEMS repository (a few seconds).
BUILD_WITH_PIXI = False

BACKUP_URL = ("https://raw.githubusercontent.com/AntaresSimulatorTeam/GEMS/main/doc/examples/notebooks/"
              "tutorial-two-pypsa-eur/pypsa-eur/resources/networks/base_s_1_elec.nc")

# Configuration for the PyPSA-Eur workflow (used only when BUILD_WITH_PIXI = True)
PYPSA_EUR_CONFIG = """\
benchmark:
  enabled: false

countries:
  - FR

scenario:
  clusters: [1]
  opts: [""]
  sector_opts: [""]
  planning_horizons: [2050]

snapshots:
  start: "2013-01-01"
  end: "2013-01-03"
  inclusive: left

solving:
  solver:
    name: highs
    options: "highs-default"
  mem_mb: 10000 # 10 GB RAM max

atlite:
  nprocesses: 2 # number of parallel processes for atlite 

sector:
  hydrogen_underground_storage_locations:
    - onshore
  regional_co2_sequestration_potential:
    enable: false
  district_heating:
    supply_temperature_approximation:
      rolling_window_ambient_temperature: 48

plotting:
  costs_threshold: 0.001
  energy_threshold: 0.1
"""

def download_prebuilt():
    network_file.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["curl", "-fsSL", "--max-time", "120", "-o", str(network_file), BACKUP_URL], check=True)
    print(f"Downloaded pre-generated network: {network_file} ({network_file.stat().st_size / 1e3:.0f} kB)")

def build_with_pixi():
    # PyPSA-Eur in a short path (Snakemake creates very long file names)
    pypsa_eur_dir = WORKDIR / "pypsa-eur-src"
    target = "resources/networks/base_s_1_elec.nc"
    # 1. Install Pixi (PyPSA-Eur package manager)
    pixi = shutil.which("pixi") or str(Path.home() / ".pixi/bin/pixi")
    if not Path(pixi).is_file():
        subprocess.run(["bash", "-c", "curl -fsSL https://pixi.sh/install.sh | bash"], check=True)
    print(f"Pixi: {pixi}")
    # 2. Clone PyPSA-Eur v2026.08.0 (last release producing base_s_{clusters}_elec.nc)
    if not pypsa_eur_dir.is_dir():
        subprocess.run(["git", "clone", "--branch", "v2026.08.0", "--depth", "1",
                        "https://github.com/PyPSA/pypsa-eur.git", str(pypsa_eur_dir)], check=True)
    # 3. Install the PyPSA-Eur environment (reads pixi.toml)
    subprocess.run([pixi, "install"], cwd=pypsa_eur_dir, check=True)
    # 4. Configure the study: France, 1 node, 2 days (merged with config.default.yaml)
    (pypsa_eur_dir / "config/config.yaml").write_text(PYPSA_EUR_CONFIG, encoding="utf-8")
    # 5. Run Snakemake ONLY up to the target file: it retrieves just the datasets this file needs
    #    (no sector coupling, no solving). --rerun-incomplete resumes after an interruption.
    subprocess.run([pixi, "run", "snakemake", "-c", "all", "-j", "2", "--rerun-incomplete", target],
                   cwd=pypsa_eur_dir, check=True)
    network_file.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(pypsa_eur_dir / target, network_file)
    print(f"PyPSA-Eur network generated: {network_file}")

# If the network is not available, build it with Pixi or download the pre-generated copy
if not network_file.is_file():
    if BUILD_WITH_PIXI:
        try:
            build_with_pixi()
        except (subprocess.CalledProcessError, OSError) as err:
            print(f"PyPSA-Eur generation failed ({err}); falling back to the pre-generated network.")
            download_prebuilt()
    else:
        download_prebuilt()
else:
    print(f"Network already available: {network_file}")

## Part 1 - Loading the network

### 1.1 Loading the network

Results are stored as NetCDF files under `pypsa-eur/results/networks/`.
The filename encodes the configuration: `base_s_***.nc`

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import pypsa

n = pypsa.Network(str(network_file))
print(n)

# Conversion to GEMS format

## Part 2 - Imports & Logging

Firstly we import the Python package to convert PyPSA studies ; [pypsa-to-gems-converter](https://pypi.org/project/pypsa-to-gems-converter/).

In [ ]:
import sys
import src as _pypsa_to_gems_converter_pkg
sys.modules.setdefault("pypsa_to_gems_converter", _pypsa_to_gems_converter_pkg)

import logging
from pathlib import Path
from pypsa import Network

# Configure logging so progress is visible in the notebook
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(name)s – %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("pypsa_to_gems")

study_dir = Path("tmp/my_study")  # Absolute path to the GEMS study directory

PyPSA_file_path = str(network_file)  # Absolute path to the PyPSA file
network = Network(PyPSA_file_path)
print(network)  # Print the network to verify it loaded correctly

## Part 3 - Conversion PYPSA network to GEMS

Now, we run the converter with our PyPSA sample study :

In [ ]:
from pypsa_to_gems_converter import PyPSAStudyConverter
# Convert PyPSA network to GEMS
PyPSAStudyConverter(
    pypsa_network=network,
    study_dir=study_dir,
    series_file_format=".tsv",  # Supported formats: .tsv, .csv
    solver_name="coin",  # Investment studies only support 'coin' and 'xpress'
).to_gems_study()

## Part 4 - Inspect the GEMS Study

This part focuses on the results analysis of the converted study :

In [ ]:
import yaml
from collections import defaultdict

gems_study_file_path = Path(study_dir) / "systems" / "input"

system_yml_files = list(gems_study_file_path.rglob("system.yml"))
if not system_yml_files:
    print("system.yml not found.")
else:
    data = yaml.safe_load(system_yml_files[0].read_text())["system"]

    components = data.get("components", [])
    connections = data.get("connections", [])

    # Group components by model type
    by_model = defaultdict(list)
    for c in components:
        by_model[c["model"]].append(c["id"])

    print(f"System : {data['id']}")
    print(f"\nComponents ({len(components)} total)")
    print("-" * 20)
    for model, ids in sorted(by_model.items()):
        short = model.split(".")[-1]
        print(f"  [{short}]  ({len(ids)})")
        for cid in ids:
            print(f"    - {cid}")

    print(f"\nConnections ({len(connections)} total)")
    print("-" * 20)
    for conn in connections:
        print(f"  {conn.get('component1')}.{conn.get('port1')}  →  {conn.get('component2')}.{conn.get('port2')}")

## Part 5 - Run the GEMS optimization

In this section, we use [GemsPy](https://gemspy.readthedocs.io/en/latest/) to run the GEMS converted simulation.

In [ ]:
from gems_craft.study.folder import load_study
from gems_runner.session.session import SimulationSession
from gems_craft.optim_config.parsing import OptimConfig, TimeScopeConfig, ScenarioScopeConfig

n_timesteps = len(network.snapshots)

print("STUDY LOADING")
study = load_study(study_dir / "systems")
print("\tStudy loaded")

optim_config = OptimConfig(
    time_scope=TimeScopeConfig(first_time_step=0, last_time_step=n_timesteps - 1),
    scenario_scope=ScenarioScopeConfig(include=[0]),
)

print("\nSOLVING OPTIMIZATION PROBLEM")
result = SimulationSession(study=study, optim_config=optim_config).run()
print("\tOptimization problem solved")

## Part 6 - Analyse the simulation results

The results are returned as a `SimulationTable` object by GemsPy.  
Below we extract the long-format DataFrame and visualise:
- the **objective value** (total system cost)
- the **installed capacity** (`p_nom`) per generator technology
- the **dispatch** (`p`) time series per technology

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = result.data

# ── Objective value ────────────────────────────────────────────────────────────
obj_row = df[df["output"] == "objective-value"]
if not obj_row.empty:
    obj_value = float(obj_row["value"].iloc[0])
    print(f"Objective value (total system cost): {obj_value:,.0f} €")

# ── Installed capacity (p_nom) per generator technology ───────────────────────
p_nom_df = df[
    (df["output"] == "p_nom") &
    (df["component"].str.startswith("generator_", na=False))
].copy()

p_nom_df["technology"] = p_nom_df["component"].str.split("_").str[-1]
capacity = p_nom_df.groupby("technology")["value"].sum().sort_values(ascending=False)
capacity = capacity[capacity > 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(capacity.index, capacity.values / 1e3, color="steelblue")
axes[0].set_xlabel("Installed capacity (GW)")
axes[0].set_title("Generator installed capacity (p_nom)")
axes[0].invert_yaxis()

# ── Dispatch time series (p) per technology ───────────────────────────────────
p_df = df[
    (df["output"] == "p") &
    (df["component"].str.startswith("generator_", na=False)) &
    (df["absolute_time_index"].notna())
].copy()

p_df["technology"] = p_df["component"].str.split("_").str[-1]
p_df["t"] = pd.to_numeric(p_df["absolute_time_index"])

dispatch = (
    p_df.groupby(["t", "technology"])["value"]
    .sum()
    .unstack("technology")
    .fillna(0)
    .clip(lower=0)
)
dispatch = dispatch / 1e3

dispatch.plot.area(ax=axes[1], linewidth=0)
axes[1].set_xlabel("Time step")
axes[1].set_ylabel("Generation (GW)")
axes[1].set_title("Dispatch by technology")
axes[1].legend(loc="upper right", fontsize=7, ncol=2)

plt.tight_layout()
plt.show()

## Part 7 - Validate the transfer

To verify that the conversion is faithful, we solve the original PyPSA network and compare it with the GEMS result: total cost first, then the optimal capacity per carrier to localise any gap.

In [ ]:
network.optimize(solver_name="highs")
# PyPSA keeps the fixed cost of existing non-extendable capacity apart (objective_constant); GEMS includes it.
pypsa_obj = network.objective + network.objective_constant
print(f"GEMS total cost  : {obj_value:,.0f} €")
print(f"PyPSA total cost : {pypsa_obj:,.0f} €")
print(f"Relative gap     : {abs(obj_value - pypsa_obj) / pypsa_obj:.1e}")

# Per-carrier comparison of the optimal capacities (localises any gap in the objective)
pypsa_cap = network.generators.groupby("carrier")["p_nom_opt"].sum()
gems_cap = (df[(df["output"] == "p_nom") & df["component"].str.startswith("generator_", na=False)]
            .assign(carrier=lambda d: d["component"].str.split("_").str[-1])
            .groupby("carrier")["value"].sum())
cap_cmp = pd.concat({"PyPSA (MW)": pypsa_cap, "GEMS (MW)": gems_cap}, axis=1).fillna(0)
cap_cmp["abs diff (MW)"] = (cap_cmp["PyPSA (MW)"] - cap_cmp["GEMS (MW)"]).abs()
print(cap_cmp.round(1))
print("Max capacity difference (MW):", round(cap_cmp["abs diff (MW)"].max(), 3))